# 面试题整理（Python / 金融工程 / 算法）

本 Notebook 将题目按主题结构化整理，并为 Coding 题提供可运行 Python 示例。

## 1. Python 布尔运算与元组特性

In [ ]:
result = (False, 0)
if result:
    print('Stay positive!')

def redundant_check():
    print("This won't be called")
    return True

print(True or redundant_check())

## 2. 债券现值（PV）计算

$PV = \sum_{t=1}^{n} \frac{CF_t}{(1+r_t)^t}$

In [ ]:
def calculate_bond_pv(face_value, coupon_rate, periods, discount_rates):
    if len(discount_rates) != periods:
        raise ValueError('贴现率数组长度必须等于期数')
    coupon_payment = face_value * coupon_rate
    pv = 0
    for t in range(1, periods + 1):
        cash_flow = coupon_payment + (face_value if t == periods else 0)
        pv += cash_flow / ((1 + discount_rates[t-1]) ** t)
    return round(pv, 2)

print(calculate_bond_pv(1000, 0.05, 3, [0.02, 0.025, 0.03]))

## 3. 实现订单簿（Order Book）

In [ ]:
import bisect

class OrderBook:
    def __init__(self):
        self.bids = []
        self.asks = []

    def limit_order(self, side, price, quantity):
        if side == 'BUY':
            while quantity > 0 and self.asks and price >= self.asks[0][0]:
                ask_price, ask_qty = self.asks.pop(0)
                if ask_qty > quantity:
                    self.asks.insert(0, (ask_price, ask_qty - quantity))
                    quantity = 0
                else:
                    quantity -= ask_qty
            if quantity > 0:
                bisect.insort(self.bids, (price, quantity))
        elif side == 'SELL':
            while quantity > 0 and self.bids and price <= self.bids[-1][0]:
                bid_price, bid_qty = self.bids.pop(-1)
                if bid_qty > quantity:
                    self.bids.append((bid_price, bid_qty - quantity))
                    quantity = 0
                else:
                    quantity -= bid_qty
            if quantity > 0:
                bisect.insort(self.asks, (price, quantity))

    def __repr__(self):
        return f'Asks: {self.asks}\nBids: {self.bids[::-1]}'

ob = OrderBook()
ob.limit_order('BUY', 100, 10)
ob.limit_order('SELL', 99, 5)
print(ob)

## 4. 时间复杂度分析与优化

- 识别复杂度来源：嵌套循环、递归深度、分治。
- 优化：`dict/set`、`lru_cache`、NumPy 向量化。

## 5. Python 垃圾回收机制（GC）

- 引用计数
- 分代回收
- 标记-清除

## 6. 行为面试：应对紧急且陌生的任务

确认需求 → 资源盘点 → 主动沟通 → 按时交付。

## 7. 翻转嵌套字典

In [ ]:
def reverse_nested_dict(d):
    reversed_dict = {}
    for outer_key, inner_map in d.items():
        if isinstance(inner_map, dict):
            for inner_key, value in inner_map.items():
                if inner_key not in reversed_dict:
                    reversed_dict[inner_key] = {}
                reversed_dict[inner_key][outer_key] = value
    return reversed_dict

print(reverse_nested_dict({'Strategy_A': {'Pnl': 100, 'Sharpe': 2.1}, 'Strategy_B': {'Pnl': 200, 'Sharpe': 1.8}}))

## 8. 任务调度器（Task Scheduler）

In [ ]:
from collections import Counter

def leastTime(tasks, n):
    counts = Counter(tasks)
    max_freq = max(counts.values())
    max_freq_count = sum(1 for c in counts.values() if c == max_freq)
    return max((max_freq - 1) * (n + 1) + max_freq_count, len(tasks))

print(leastTime(['A','A','A','B','B','B'], 2))

## 9. 文件保护数据迁移（File Protection）

In [ ]:
def getMinimumChanges(fileSize, minSize):
    diff = [fileSize[i] - minSize[i] for i in range(len(fileSize))]
    if sum(diff) < 0:
        return -1
    plus = sorted([x for x in diff if x > 0], reverse=True)
    minus = sorted([x for x in diff if x < 0])
    if not minus:
        return 0
    changes = len(minus)
    needed = abs(sum(minus))
    cur = 0
    for p in plus:
        cur += p
        changes += 1
        if cur >= needed:
            break
    return changes

print(getMinimumChanges([4,1,5,2,3],[3,2,2,1,4]))

## 10. 五局三胜制获胜概率（DP / 概率论）

In [ ]:
def win_probability(p, target_wins=3):
    memo = {}
    def solve(i, j):
        if i == 0:
            return 1.0
        if j == 0:
            return 0.0
        if (i, j) in memo:
            return memo[(i, j)]
        memo[(i, j)] = p * solve(i - 1, j) + (1 - p) * solve(i, j - 1)
        return memo[(i, j)]
    return solve(target_wins, target_wins)

print(f'{win_probability(0.6, 3):.4f}')